# Example 9.5.2 / 2: Use the 2D Layout

Companion notebook for **9.5.2 Excel Spreadsheets** in the AMPL lab Documentation, section "Use the 2D Layout". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Use the 2D Layout

A relational sheet stores one record per row, so data indexed by two sets uses two key
columns. The `2D` option instead puts one index down the first column and the other across the
header row, the way people usually lay out such data in a spreadsheet. A `2D` declaration lists
exactly one parameter. When writing, the first set in the key labels the rows and the second
the columns:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
set PERIOD;
param market {PROD, PERIOD} >= 0;

table Market2D OUT "amplxl" "market2d.xlsx" "Market" "2D" "backup=false":
    [product, period], market;
table Market2DIn IN "amplxl" "market2d.xlsx" "Market" "2D":
    [product, period], market;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.set["PERIOD"] = [1, 2]
ampl.param["market"] = {("bands", 1): 6000, ("bands", 2): 5000,
                        ("coils", 1): 4000, ("coils", 2): 4500,
                        ("plate", 1): 3500, ("plate", 2): 3000}

In [5]:
# run.py
ampl.eval("load amplxl.dll;")
ampl.write_table("Market2D")
ampl.eval("reset data market;")
ampl.read_table("Market2DIn")
print(ampl.param["market"].to_pandas().unstack())

       market      
index1      1     2
index0             
bands    6000  5000
coils    4000  4500
plate    3500  3000


### Pure AMPL: data.dat and model.run

The sheet `Market` has a `product` column of row labels and one column per period. When
reading, `amplxl` takes the column headed by a key name as the row labels and every other
column heading as a member of the second set. Prefer the long relational layout for interchange
with databases; prefer `2D` when it materially improves the workbook's human use.

Pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
set PERIOD := 1 2;
param market:  1     2 :=
  bands      6000  5000
  coils      4000  4500
  plate      3500  3000 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
load amplxl.dll;
write table Market2D;
reset data market;
read table Market2DIn;
display market;

market :=
bands 1   6000
bands 2   5000
coils 1   4000
coils 2   4500
plate 1   3500
plate 2   3000
;

